<a href="https://colab.research.google.com/github/gabrielaaku/sna-gastronomi-nusantara-kelompok7/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
   !pip install requests beautifulsoup4

## Scraping resep rendang

In [ ]:

import requests
from bs4 import BeautifulSoup
import csv
import json
import re
import time
import os
from itertools import combinations

# ======================= CONFIG =======================

BASE_URL = "https://cookpad.com"
SEARCH_KEYWORDS = ["rendang"]
PAGES_PER_KEYWORD = 3
TARGET_RECIPES_PILOT = 60
RATE_LIMIT_DELAY = 2.0

HEADERS = {
    "User-Agent": "Mozilla/5.0 (riset-akademik; kontak: biankarosa02@gmail.com) "
                  "- tugas kuliah Analisis Jejaring Sosial, non-komersial"
}

OUT_DIR = "output"
RAW_JSON = os.path.join(OUT_DIR, "recipes_raw.json")
NODES_CSV = os.path.join(OUT_DIR, "nodes.csv")
EDGES_CSV = os.path.join(OUT_DIR, "edges.csv")

# ========================================================

def check_robots_txt() -> None:
    try:
        r = requests.get(f"{BASE_URL}/robots.txt", headers=HEADERS, timeout=10)
        print("--- robots.txt cookpad.com (ringkas) ---")
        for line in r.text.splitlines():
            if line.strip().startswith(("User-agent: *", "Disallow", "Allow")):
                print(" ", line)
            if line.strip() == "":
                break
        print("-> Path /id/cari/ dan /id/resep/ TIDAK ada di daftar Disallow untuk "
              "User-agent: * -> aman untuk discraping script ini.\n")
    except requests.RequestException as e:
        print(f"Gagal ambil robots.txt: {e}. Cek manual dulu sebelum lanjut scraping.")

def clean_ingredient_name(raw_name):
    text = raw_name.strip()
    if not text or text.endswith(":"):
        return None
    text = text.lower()
    text = text.split(",")[0]
    text = re.sub(r"\(.*?\)", "", text)
    UNITS = r"\b(sdm|sdt|gr|gram|kg|ml|liter|butir|siung|buah|ruas|lembar|batang|ikat|ons|cc|bh|lbr|btg|btr)\b"
    text = re.sub(UNITS, " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if len(text) > 1 else None

def fetch_soup(url: str):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return BeautifulSoup(r.text, "html.parser")
    except requests.RequestException as e:
        print(f"  [gagal] {url} -> {e}")
        return None

def collect_recipe_links():
    links = set()
    for keyword in SEARCH_KEYWORDS:
        for page in range(1, PAGES_PER_KEYWORD + 1):
            url = f"{BASE_URL}/id/cari/{keyword}?page={page}"
            print(f"Ambil daftar resep: {url}")
            soup = fetch_soup(url)
            time.sleep(RATE_LIMIT_DELAY)
            if soup is None:
                continue
            for a in soup.select('a.block-link__main[href*="/id/resep/"]'):
                href = a.get("href")
                if href and href != "/id/resep/baru":
                    links.add(BASE_URL + href)
    print(f"Total link resep unik ditemukan: {len(links)}\n")
    return list(links)

def scrape_recipe(url: str):
    soup = fetch_soup(url)
    if soup is None:
        return None
    title_tag = soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else url
    items = soup.select('.ingredient-list li[id^="ingredient_"]')
    cleaned = []
    for li in items:
        span = li.find("span")
        if span is None:
            continue
        cleaned_name = clean_ingredient_name(span.get_text(" ", strip=True))
        if cleaned_name:
            cleaned.append(cleaned_name)
    if not cleaned:
        print(f"  [peringatan] Tidak ada bahan terbaca di {url}")
        return None
    return {"url": url, "title": title, "ingredients": cleaned}

def load_existing_raw():
    if os.path.exists(RAW_JSON):
        with open(RAW_JSON, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_raw(recipes):
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(RAW_JSON, "w", encoding="utf-8") as f:
        json.dump(recipes, f, ensure_ascii=False, indent=2)

def build_network(recipes):
    node_ids = {}
    edge_weights = {}

    def get_or_create_id(name):
        if name not in node_ids:
            node_ids[name] = len(node_ids) + 1
        return node_ids[name]

    for recipe in recipes:
        ids_in_recipe = sorted({get_or_create_id(ing) for ing in recipe["ingredients"]})
        for a, b in combinations(ids_in_recipe, 2):
            edge_weights[(a, b)] = edge_weights.get((a, b), 0) + 1

    os.makedirs(OUT_DIR, exist_ok=True)

    with open(NODES_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        for name, node_id in sorted(node_ids.items(), key=lambda x: x[1]):
            writer.writerow([node_id, name])

    with open(EDGES_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["source", "target", "weight"])
        for (a, b), w in edge_weights.items():
            writer.writerow([a, b, w])

    print(f"\nSelesai. {len(node_ids)} node (bahan unik), {len(edge_weights)} edge "
          f"(pasangan bahan) dari {len(recipes)} resep.")
    print(f"Tersimpan di: {NODES_CSV} dan {EDGES_CSV}")

    if len(node_ids) < 50:
        print("\n[PERHATIAN] Node masih di bawah 50 (syarat minimum bukti pilot "
              "proposal). Tambah PAGES_PER_KEYWORD atau SEARCH_KEYWORDS lalu run lagi.")

def main():
    check_robots_txt()
    recipes = load_existing_raw()
    print(f"Sudah ada {len(recipes)} resep tersimpan sebelumnya (checkpoint).\n")
    scraped_urls = {r["url"] for r in recipes}
    links = [l for l in collect_recipe_links() if l not in scraped_urls]

    for link in links:
        if len(recipes) >= TARGET_RECIPES_PILOT:
            print("Target pilot tercapai, berhenti.")
            break
        print(f"[{len(recipes)+1}/{TARGET_RECIPES_PILOT}] Scraping: {link}")
        data = scrape_recipe(link)
        if data:
            recipes.append(data)
        time.sleep(RATE_LIMIT_DELAY)
        if len(recipes) % 10 == 0:
            save_raw(recipes)
            print(f"  -> checkpoint tersimpan ({len(recipes)} resep)\n")

    save_raw(recipes)
    build_network(recipes)

main()

--- robots.txt cookpad.com (ringkas) ---
  User-agent: *
  Disallow: /user/confirm_premium_navi
  Disallow: /*?_pxhc=*
  Disallow: /*/recipes/*/reactions
  Disallow: /*/recipes/*/similar_recipes
  Disallow: /*/recipes/*/newer_premium_recipes
  Disallow: /*/recipes/*/cooksnaps_section
  Disallow: /*/me/recipes/*/bookmark_folders
  Disallow: /*/recipes/*/print
  Disallow: /*/activity_logs
  Disallow: /*/service_feedbacks
  Allow: /
-> Path /id/cari/ dan /id/resep/ TIDAK ada di daftar Disallow untuk User-agent: * -> aman untuk discraping script ini.

Sudah ada 60 resep tersimpan sebelumnya (checkpoint).

Ambil daftar resep: https://cookpad.com/id/cari/rendang?page=1
Ambil daftar resep: https://cookpad.com/id/cari/rendang?page=2
Ambil daftar resep: https://cookpad.com/id/cari/rendang?page=3
Total link resep unik ditemukan: 90

Target pilot tercapai, berhenti.

Selesai. 233 node (bahan unik), 2343 edge (pasangan bahan) dari 60 resep.
Tersimpan di: output/nodes.csv dan output/edges.csv


In [ ]:
import shutil
shutil.make_archive("dataset_rendang", "zip", "output")

'/content/dataset_rendang.zip'

In [ ]:
# library yang dipakai
import sys, requests, bs4, pandas as pd, networkx as nx
print("Python       :", sys.version.split()[0])
print("requests     :", requests.__version__)
print("beautifulsoup:", bs4.__version__)
print("pandas       :", pd.__version__)
print("networkx     :", nx.__version__)

Python       : 3.13.18
requests     : 2.32.4
beautifulsoup: 4.13.5
pandas       : 2.2.3
networkx     : 3.7


## Statistik awal jaringan rendang

In [ ]:
# statistik jaringan dari data rendang
import os, json
import pandas as pd, networkx as nx

base = "output" if os.path.exists("output/nodes.csv") else "."
nodes = pd.read_csv(f"{base}/nodes.csv")
edges = pd.read_csv(f"{base}/edges.csv")
recipes = json.load(open(f"{base}/recipes_raw.json", encoding="utf-8"))

G = nx.Graph()
G.add_nodes_from(nodes["id"])
for s, t, w in edges[["source", "target", "weight"]].itertuples(index=False):
    G.add_edge(s, t, weight=w)

lcc = G.subgraph(max(nx.connected_components(G), key=len))
label = dict(zip(nodes["id"], nodes["label"]))
dup = len(recipes) - len({(r["title"], tuple(r["ingredients"])) for r in recipes})

stats = {
    "jumlah resep": len(recipes),
    "resep berisi identik (duplikat)": dup,
    "rata-rata bahan per resep": round(sum(len(r["ingredients"]) for r in recipes) / len(recipes), 1),
    "jumlah simpul": G.number_of_nodes(),
    "jumlah sisi": G.number_of_edges(),
    "density": round(nx.density(G), 3),
    "komponen terhubung": nx.number_connected_components(G),
    "diameter (komponen terbesar)": nx.diameter(lcc),
    "rata-rata panjang jalur": round(nx.average_shortest_path_length(lcc), 2),
    "rata-rata clustering coefficient": round(nx.average_clustering(G), 3),
}
display(pd.DataFrame([(k, str(v)) for k, v in stats.items()], columns=["Ukuran", "Nilai"]))

top = sorted(G.degree, key=lambda x: -x[1])[:10]
display(pd.DataFrame([(label[n], d) for n, d in top], columns=["bahan", "degree"]))

,Ukuran,Nilai
0,jumlah resep,60
1,resep berisi identik (duplikat),1
2,rata-rata bahan per resep,9.1
3,jumlah simpul,233
4,jumlah sisi,2343
5,density,0.087
6,komponen terhubung,1
7,diameter (komponen terbesar),4
8,rata-rata panjang jalur,2.22
9,rata-rata clustering coefficient,0.867


,bahan,degree
0,air,123
1,bawang putih,116
2,daun salam,115
3,daun jeruk,111
4,bawang merah,108
5,garam,107
6,lengkuas,100
7,kemiri,85
8,jahe,82
9,minyak goreng,79


## Scraping resep dari daerah lain
Gudeg, rawon, woku, ayam betutu, dan soto betawi, masing-masing 10 resep.


In [ ]:
import os, json, time
from datetime import datetime, timezone
from urllib.parse import quote

# kata kunci pencarian -> daerah asal masakan
MASAKAN_DAERAH = {
    "gudeg": "DI Yogyakarta",
    "rawon": "Jawa Timur",
    "woku": "Sulawesi Utara",
    "ayam betutu": "Bali",
    "soto betawi": "DKI Jakarta",
}
RESEP_PER_MASAKAN = 10
MULTI_DIR = "output_multidaerah"
MULTI_JSON = os.path.join(MULTI_DIR, "recipes_multidaerah.json")
os.makedirs(MULTI_DIR, exist_ok=True)

def links_untuk(keyword, halaman=1):
    url = f"{BASE_URL}/id/cari/{quote(keyword)}?page={halaman}"
    print(f"Ambil daftar resep: {url}")
    soup = fetch_soup(url)
    time.sleep(RATE_LIMIT_DELAY)
    if soup is None:
        return []
    hasil = []
    for a in soup.select('a.block-link__main[href*="/id/resep/"]'):
        href = a.get("href")
        if href and href != "/id/resep/baru" and BASE_URL + href not in hasil:
            hasil.append(BASE_URL + href)
    return hasil

data = json.load(open(MULTI_JSON, encoding="utf-8")) if os.path.exists(MULTI_JSON) else []
sudah = {r["url"] for r in data}

for keyword, daerah in MASAKAN_DAERAH.items():
    jumlah = sum(1 for r in data if r["keyword"] == keyword)
    if jumlah >= RESEP_PER_MASAKAN:
        print(f"[{keyword}] sudah {jumlah} resep, dilewati.")
        continue
    for link in links_untuk(keyword):
        if jumlah >= RESEP_PER_MASAKAN:
            break
        if link in sudah:
            continue
        r = scrape_recipe(link)
        time.sleep(RATE_LIMIT_DELAY)
        if r:
            r.update({
                "keyword": keyword,
                "daerah": daerah,
                "scraped_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            })
            data.append(r)
            sudah.add(link)
            jumlah += 1
            print(f"  [{keyword} {jumlah}/{RESEP_PER_MASAKAN}] {r['title']}")
    json.dump(data, open(MULTI_JSON, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

print("\nRingkasan per masakan:")
for keyword, daerah in MASAKAN_DAERAH.items():
    rs = [r for r in data if r["keyword"] == keyword]
    bahan = {b for r in rs for b in r["ingredients"]}
    print(f"  {keyword:12} ({daerah:15}): {len(rs):2} resep, {len(bahan):3} label bahan mentah")
print(f"Total: {len(data)} resep -> {MULTI_JSON}")

[gudeg] sudah 10 resep, dilewati.
[rawon] sudah 10 resep, dilewati.
[woku] sudah 10 resep, dilewati.
[ayam betutu] sudah 10 resep, dilewati.
[soto betawi] sudah 10 resep, dilewati.

Ringkasan per masakan:
  gudeg        (DI Yogyakarta  ): 10 resep,  87 label bahan mentah
  rawon        (Jawa Timur     ): 10 resep,  98 label bahan mentah
  woku         (Sulawesi Utara ): 10 resep,  74 label bahan mentah
  ayam betutu  (Bali           ): 10 resep,  94 label bahan mentah
  soto betawi  (DKI Jakarta    ): 10 resep, 115 label bahan mentah
Total: 50 resep -> output_multidaerah/recipes_multidaerah.json


## Penyeragaman nama bahan
Satu bahan sering ditulis berbeda-beda, misalnya "santan kental", "santan kara", dan "santan rose brand". Semuanya disatukan menjadi "santan".

In [ ]:
import os, re, json, csv
from itertools import combinations
from collections import Counter
import pandas as pd
import networkx as nx

def _find(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

PILOT_JSON = _find(["output/recipes_raw.json", "recipes_raw.json"])
MULTI_JSON = _find(["output_multidaerah/recipes_multidaerah.json"])
CLEAN_DIR = "output_clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

# judul bagian resep, bukan bahan
NON_BAHAN = {
    "bumbu halus", "tambahan", "pelengkap", "bahan lainnya", "aromatik",
    "bahan adonan tepung", "seasoning", "putih", "sekitar", "bahan",
    "bumbu", "bumbu cemplung", "bahan utama", "bumbu iris", "isian",
    "resep isian ayam suwir",
}

# judul bagian resep dengan pola tertentu
NON_BAHAN_POLA = re.compile(r"^(bahan|resep|topping|pelengkap|hijau$|bumbu (tambahan|aromatik|cemplung|dihaluskan|kering|chopper|halus))")

# merek, takaran, dan cara potong, dihapus
BUANG = [
    r"\brose brand\b", r"\bsajiku\b", r"\bbamboe\b", r"\bsasa\b", r"\bkara\b",
    r"\bmunik\b", r"\bkeraton\b", r"\basatu\b", r"\bmasako\b", r"\broyco\b",
    r"\bsecukupnya\b", r"\bsesuai selera\b", r"\buntuk\b.*$", r"\btambahkan\b.*$",
    r"\bsampai\b.*$", r"\bbagian\b.*$", r"\bdari\b.*$", r"\bjika\b.*$",
    r"\bukuran\b", r"\bbesar\b", r"\bkecil\b", r"\bsedang\b", r"\bpotong\b",
    r"\bdadu\b", r"\bcincang\b", r"\bgeprek\b", r"\bdigeprek\b", r"\biris\b",
    r"\bmemarkan\b", r"\bsimpulkan\b", r"\bdisobek\b.*$", r"\bsiap pakai\b",
    r"\bkemasan\b", r"\bsachet\b", r"\bbungkus\b", r"\bplastik\b",
    r"\binstant?\b", r"\bpraktis\b", r"\bml\b", r"\bgr\b", r"\bgram\b",
    r"\bmurni\b", r"\bperasan pertama\b", r"\bperasan kedua\b", r"\bperas\b",
    r"\bjadi\b", r"\btanpa lemak\b", r"\bhas dalam\b", r"\bsengkel\b",
    r"\btop ?side\b", r"\bgandik\b", r"\bsetan\b",
]

# sinonim -> pakai nama baku
SINONIM = [
    (r"santan|kelapa.*santan", "santan"),
    (r"air asam|asam jawa", "asam jawa"),
    (r"air kaldu|kaldu|penyedap|micin|msg|rasa (ayam|sapi)", "kaldu bubuk"),
    (r"^air\b", "air"),
    (r"base genep", "base genep"),
    (r"^rendang$|^sajiku|^bumbu biji|bumbu.*(rendang|rawon|soto|racik|instan|basah pasar|rempah|minang)|rendang.*bumbu", "bumbu jadi"),
    (r"serai|sereh|^sere$", "serai"),
    (r"laos|lengkuas", "lengkuas"),
    (r"daun jeruk", "daun jeruk"),
    (r"daun salam|^salam$", "daun salam"),
    (r"daun kunyit", "daun kunyit"),
    (r"kunyit", "kunyit"),
    (r"kencur", "kencur"),
    (r"jahe", "jahe"),
    (r"kemiri", "kemiri"),
    (r"kemangi", "kemangi"),
    (r"kluwa?e?k|keluak", "kluwek"),
    (r"terasi", "terasi"),
    (r"rawit", "cabai rawit"),
    (r"cabe|cabai", "cabai merah"),
    (r"daun bawang|bawang daun|prei|prey", "daun bawang"),
    (r"bawang goreng", "bawang goreng"),
    (r"^b merah|bawang merah|^bamer$", "bawang merah"),
    (r"^b putih|bawang putih|^baput$", "bawang putih"),
    (r"bawang bomba", "bawang bombay"),
    (r"cengk[ie]h", "cengkeh"),
    (r"lada|merica", "merica"),
    (r"lawang", "bunga lawang"),
    (r"kayu ?manis", "kayu manis"),
    (r"jinten|jintan", "jintan"),
    (r"ketumbar", "ketumbar"),
    (r"^pala", "pala"),
    (r"kelapa", "kelapa parut"),
    (r"telur puyuh", "telur puyuh"),
    (r"telur itik|telur bebek", "telur bebek"),
    (r"telur", "telur"),
    (r"kornet|corned", "kornet"),
    (r"kikil", "kikil"),
    (r"daging ayam|ayam|dada|paha", "ayam"),
    (r"daging|tetelan|iga|buntut|urat", "daging sapi"),
    (r"tuna|cakalang", "ikan tuna"),
    (r"tongkol", "ikan tongkol"),
    (r"nangka|gori", "nangka muda"),
    (r"tomat", "tomat"),
    (r"taoge|tauge|kecambah", "tauge"),
    (r"seledri", "seledri"),
    (r"tahu", "tahu"),
    (r"tempe", "tempe"),
    (r"^nasi", "nasi"),
    (r"beras ketan", "beras ketan"),
    (r"beras merah", "beras merah"),
    (r"^beras", "beras"),
    (r"emping", "emping"),
    (r"sambal", "sambal"),
    (r"^teh", "teh"),
    (r"susu", "susu"),
    (r"fibercreme|krimer", "krimer"),
    (r"daun singkong", "daun singkong"),
    (r"kecap.*manis", "kecap manis"),
    (r"tepung terigu", "tepung terigu"),
    (r"maizena", "maizena"),
    (r"ragi", "ragi"),
    (r"kentang|potato", "kentang"),
    (r"jengkol", "jengkol"),
    (r"gula merah|palm sugar|gula jawa", "gula merah"),
    (r"^gula", "gula pasir"),
    (r"garam|salt", "garam"),
    (r"minyak", "minyak goreng"),
    (r"spaghetti|pasta", "pasta"),
    (r"pandan", "daun pandan"),
    (r"daun pisang", "daun pisang"),
    (r"kacang merah", "kacang merah"),
    (r"^kacang", "kacang tanah"),
]

def normalisasi(raw):
    t = raw.lower().strip()
    if t in NON_BAHAN or NON_BAHAN_POLA.search(t):
        return []
    hasil = []
    for p in re.split(r"\s+dan\s+", t):   # "garam dan kaldu" -> 2 bahan
        for pola in BUANG:
            p = re.sub(pola, " ", p)
        p = re.sub(r"\s+", " ", p).strip()
        if not p or p in NON_BAHAN or NON_BAHAN_POLA.search(p) or len(p) < 2:
            continue
        for pola, baku in SINONIM:
            if re.search(pola, p):
                p = baku
                break
        hasil.append(p)
    return hasil

def muat(path):
    if not path:
        return []
    data = json.load(open(path, encoding="utf-8"))
    for r in data:                              # data rendang belum punya label daerah
        r.setdefault("keyword", "rendang")
        r.setdefault("daerah", "Sumatera Barat")
    return data

resep = muat(PILOT_JSON) + muat(MULTI_JSON)
print(f"Resep dimuat: {len(resep)} (pilot: {len(muat(PILOT_JSON))}, multi-daerah: {len(muat(MULTI_JSON))})")

# resep dengan judul dan bahan yang sama persis dianggap duplikat
unik, sudah = [], set()
for r in resep:
    kunci = (r["title"].strip().lower(), tuple(r["ingredients"]))
    if kunci not in sudah:
        sudah.add(kunci)
        unik.append(r)
print(f"Resep duplikat dibuang: {len(resep) - len(unik)}")
resep = unik

mapping = {}
for r in resep:
    bersih = []
    for b in r["ingredients"]:
        hasil = normalisasi(b)
        mapping[b] = ", ".join(hasil) if hasil else "(dibuang)"
        bersih.extend(hasil)
    r["bahan_bersih"] = sorted(set(bersih))

pd.DataFrame(sorted(mapping.items()), columns=["label_mentah", "label_bersih"]) \
  .to_csv(f"{CLEAN_DIR}/mapping_bahan.csv", index=False)

Resep dimuat: 110 (pilot: 60, multi-daerah: 50)
Resep duplikat dibuang: 1


## Jaringan setelah dibersihkan

In [ ]:
frek = Counter(b for r in resep for b in r["bahan_bersih"])
daerah_bahan = {}
for r in resep:
    for b in r["bahan_bersih"]:
        daerah_bahan.setdefault(b, Counter())[r["daerah"]] += 1

node_id = {b: i + 1 for i, b in enumerate(sorted(frek))}
bobot = Counter()
for r in resep:
    for a, b in combinations(sorted(node_id[x] for x in r["bahan_bersih"]), 2):
        bobot[(a, b)] += 1

with open(f"{CLEAN_DIR}/nodes_clean.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "label", "frekuensi", "daerah_dominan"])
    for b, i in node_id.items():
        w.writerow([i, b, frek[b], daerah_bahan[b].most_common(1)[0][0]])

with open(f"{CLEAN_DIR}/edges_clean.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["source", "target", "weight"])
    for (a, b), wt in sorted(bobot.items()):
        w.writerow([a, b, wt])

mentah = {b for r in resep for b in r["ingredients"]}
G = nx.Graph()
G.add_nodes_from(node_id.values())
G.add_weighted_edges_from((a, b, wt) for (a, b), wt in bobot.items())
lcc = G.subgraph(max(nx.connected_components(G), key=len))
ringkas = pd.DataFrame([(k, str(v)) for k, v in [
    ("jumlah resep (setelah buang duplikat)", len(resep)),
    ("label bahan mentah", len(mentah)),
    ("label dibuang (bukan bahan)", sum(v == "(dibuang)" for v in mapping.values())),
    ("bahan setelah normalisasi", G.number_of_nodes()),
    ("sisi setelah normalisasi", G.number_of_edges()),
    ("density", round(nx.density(G), 3)),
    ("diameter", nx.diameter(lcc)),
    ("rata-rata clustering", round(nx.average_clustering(G), 3)),
]], columns=["Ukuran", "Nilai"])
ringkas.to_csv(f"{CLEAN_DIR}/ringkasan_normalisasi.csv", index=False)
display(ringkas)

,Ukuran,Nilai
0,jumlah resep (setelah buang duplikat),109
1,label bahan mentah,473
2,label dibuang (bukan bahan),36
3,bahan setelah normalisasi,97
4,sisi setelah normalisasi,1751
5,density,0.376
6,diameter,3
7,rata-rata clustering,0.824


## Simpan hasil

In [ ]:
import shutil
from google.colab import files

# zip folder hasil
for folder in ["output", "output_multidaerah", "output_clean"]:
    shutil.make_archive(folder, "zip", folder)
    files.download(f"{folder}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>